In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

In [2]:
# Load and preprocess the Fashion-MNIST dataset
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

In [3]:
# Reshape the data to fit the input shape of the model (28x28 pixels, 1 color channel)
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1)).astype('float32') / 255
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1)).astype('float32') / 255

# One-hot encode the labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

In [4]:
# Define the LeNet-5 CNN model
model = models.Sequential()

# First convolutional layer (C1)
model.add(layers.Conv2D(6, (5, 5), activation='tanh', padding='same', input_shape=(28, 28, 1)))
model.add(layers.AvgPool2D((2, 2), strides=2))

# Second convolutional layer (C3)
model.add(layers.Conv2D(16, (5, 5), activation='tanh', padding='valid'))
model.add(layers.AvgPool2D((2, 2), strides=2))

# Flatten the output from the previous layer
model.add(layers.Flatten())

# Fully connected layers
model.add(layers.Dense(120, activation='tanh'))
model.add(layers.Dense(84, activation='tanh'))

# Output layer
model.add(layers.Dense(10, activation='softmax'))

/Users/davemag9/Desktop/Computer vision/venv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [6]:
# Train the model
model.fit(x_train, y_train, epochs=10, batch_size=64, validation_data=(x_test, y_test))

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - accuracy: 0.7202 - loss: 0.7814 - val_accuracy: 0.8343 - val_loss: 0.4556
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step - accuracy: 0.8512 - loss: 0.4099 - val_accuracy: 0.8564 - val_loss: 0.3971
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 15s 16ms/step - accuracy: 0.8673 - loss: 0.3586 - val_accuracy: 0.8648 - val_loss: 0.3679
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 15s 16ms/step - accuracy: 0.8809 - loss: 0.3199 - val_accuracy: 0.8756 - val_loss: 0.3421
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.8895 - loss: 0.2966 - val_accuracy: 0.8761 - val_loss: 0.3348
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.8965 - loss: 0.2791 - val_accuracy: 0.8771 - val_loss: 0.3258
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.9039 - loss: 0.2591 - val_accuracy: 0.8807 - val_loss: 0.3219
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - accuracy: 0.9078 - loss: 0.2482 - 

In [7]:
# Evaluate the model
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f'Test accuracy: {test_acc}')

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8890 - loss: 0.3130
Test accuracy: 0.8895999789237976


In [22]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tensorflow.keras.models import Model

# Ensure the model is compiled and trained first (or at least built)
model.build(input_shape=(None, 28, 28, 1))  # Explicitly specify the input shape

# Run predictions for a batch (this will call the model layers and initialize them)
_ = model.predict(np.expand_dims(x_test[0], axis=0))  # Predict one image to initialize

# Create a feature extractor that takes the output of the second-to-last layer
feature_extractor = Model(inputs=model.input, outputs=model.layers[-3].output)

# Now we can extract features
features = feature_extractor.predict(x_test)

# Select a random subset of features to speed up processing (e.g., 5000 samples)
sample_size = 5000  # Can adjust to 1000, 3000, etc.
indices = np.random.choice(x_test.shape[0], sample_size, replace=False)
features_sample = features[indices]
labels_sample = np.argmax(y_test, axis=1)[indices]

# Apply t-SNE for dimensionality reduction to 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(features_sample)

# Visualize the 2D results
plt.figure(figsize=(10, 6))
scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=labels_sample, cmap='jet', alpha=0.6)
plt.colorbar(scatter, label="Classes")
plt.title("Fashion-MNIST Feature Visualization with t-SNE")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.show()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


AttributeError: The layer sequential has never been called and thus has no defined input.